# Phase 3: Portfolio Optimization

## Project context

This notebook uses the saved daily return data from Phase 1 and the asset universe from Phase 2 to build simple long-only portfolio allocations. It does not run full backtesting or create a dashboard.

## Optimization objective

The goal is to compare three interpretable portfolio construction methods:

- Equal-weight portfolio
- Minimum-volatility portfolio
- Maximum-Sharpe portfolio

SPY is excluded from optimized asset portfolios and used only as the benchmark reference from prior phases.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import OUTPUTS_DIR, FIGURES_DIR
from src.optimization import (
    LAST_OPTIMIZATION_STATUS,
    load_returns,
    select_asset_returns,
    annualize_expected_returns,
    annualize_covariance,
    equal_weight_portfolio,
    minimum_volatility_portfolio,
    maximum_sharpe_portfolio,
    generate_random_portfolios,
    summarize_portfolio,
)
from src.visualization import (
    plot_portfolio_weights,
    plot_efficient_frontier_simulation,
    plot_portfolio_risk_return,
    plot_allocation_pie_or_bar,
)

BENCHMARK = "SPY"
RETURNS_PATH = OUTPUTS_DIR / "returns" / "daily_returns.csv"
PORTFOLIOS_DIR = OUTPUTS_DIR / "portfolios"
PORTFOLIOS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def _normalize(values):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values)
    vmin = np.nanmin(values[finite])
    vmax = np.nanmax(values[finite])
    if np.isclose(vmin, vmax):
        return np.full_like(values, 0.5, dtype=float)
    return (values - vmin) / (vmax - vmin)


def save_plotly_or_pillow(fig, output_path, chart_type, data):
    output_path = Path(output_path)
    try:
        fig.write_image(str(output_path), width=1200, height=700, scale=2)
        return "plotly"
    except Exception as exc:
        draw_basic_png(output_path, chart_type, data, str(exc))
        return "pillow_fallback"


def draw_basic_png(output_path, chart_type, data, reason):
    width, height = 1200, 700
    margin = 80
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    draw.text((margin, 24), output_path.stem.replace("_", " ").title(), fill="black", font=font)
    draw.text((margin, 48), "Rendered with Pillow fallback because Plotly PNG export was unavailable.", fill="#555555", font=font)
    left, top, right, bottom = margin, 110, width - margin, height - margin
    draw.rectangle((left, top, right, bottom), outline="#333333")

    if chart_type == "bar":
        frame = data.copy()
        values = frame["value"].astype(float).values
        labels = frame["label"].astype(str).tolist()
        scale_values = _normalize(values)
        bar_width = (right - left) / max(len(values), 1) * 0.65
        for i, value in enumerate(values):
            x = left + (right - left) * (i + 0.5) / len(values)
            y = bottom - (bottom - top) * scale_values[i]
            draw.rectangle((x - bar_width / 2, y, x + bar_width / 2, bottom), fill="#1f77b4")
            draw.text((x - 18, bottom + 10), labels[i][:8], fill="black", font=font)
            draw.text((x - 18, y - 16), f"{value:.2f}", fill="black", font=font)
    elif chart_type == "scatter":
        frame = data.copy()
        x = _normalize(frame["annualized_volatility"].values)
        y = _normalize(frame["annualized_return"].values)
        labels = frame.get("portfolio", pd.Series([""] * len(frame))).astype(str).tolist()
        for i in range(len(frame)):
            px = left + (right - left) * x[i]
            py = bottom - (bottom - top) * y[i]
            draw.ellipse((px - 4, py - 4, px + 4, py + 4), fill="#1f77b4")
            if labels[i]:
                draw.text((px + 7, py - 7), labels[i][:18], fill="black", font=font)
    image.save(output_path)


## Load daily returns

In [3]:
returns = load_returns(RETURNS_PATH)
asset_returns = select_asset_returns(returns, benchmark=BENCHMARK)
display(asset_returns.head())
print(f"Asset return shape: {asset_returns.shape}")
print(f"Date range: {asset_returns.index.min().date()} to {asset_returns.index.max().date()}")

,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA
date,,,,,,,,
2019-01-03,-0.099608,-0.036788,-0.014212,-0.007011,-0.015354,-0.015890,-0.006179,-0.060417
2019-01-04,0.042690,0.046509,0.036865,0.020410,0.036870,0.016783,0.019940,0.064068
2019-01-07,-0.002226,0.001275,0.000695,-0.004000,0.005201,-0.006415,-0.013033,0.052941
2019-01-08,0.019063,0.007251,-0.001886,0.003691,0.007271,0.023227,0.011289,-0.024895
2019-01-09,0.016981,0.014299,-0.001690,-0.016331,0.005275,-0.007926,-0.019166,0.019667


Asset return shape: (1838, 8)
Date range: 2019-01-03 to 2026-04-27


## Asset universe review

In [4]:
asset_names = asset_returns.columns.tolist()
print(asset_names)
print(f"Benchmark excluded from optimization: {BENCHMARK not in asset_names}")

['AAPL', 'MSFT', 'JPM', 'PG', 'XOM', 'JNJ', 'KO', 'NVDA']
Benchmark excluded from optimization: True


## Expected returns and covariance

In [5]:
expected_returns = annualize_expected_returns(asset_returns)
covariance_matrix = annualize_covariance(asset_returns)
display(expected_returns.to_frame("annualized_expected_return").style.format("{:.2%}"))
display(covariance_matrix.style.format("{:.4f}"))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/rovs/Library/Python/3.11/lib/python/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/rovs/Library/Python/3.11/lib/python/site-packages/ipykernel/kernelapp.py", line 736, in start
    se

AttributeError: _ARRAY_API not found

,annualized_expected_return
AAPL,30.92%
MSFT,22.74%
JPM,20.29%
PG,9.78%
XOM,16.31%
JNJ,11.22%
KO,10.19%
NVDA,76.10%


,AAPL,MSFT,JPM,PG,XOM,JNJ,KO,NVDA
AAPL,0.0951,0.0597,0.0386,0.0228,0.0272,0.0184,0.0220,0.0901
MSFT,0.0597,0.0818,0.0353,0.0202,0.0197,0.0153,0.0186,0.0951
JPM,0.0386,0.0353,0.0882,0.0184,0.0484,0.0191,0.0255,0.0528
PG,0.0228,0.0202,0.0184,0.0405,0.0129,0.0211,0.0249,0.0166
XOM,0.0272,0.0197,0.0484,0.0129,0.0966,0.0160,0.0224,0.0295
JNJ,0.0184,0.0153,0.0191,0.0211,0.0160,0.0366,0.0196,0.0090
KO,0.0220,0.0186,0.0255,0.0249,0.0224,0.0196,0.0389,0.0144
NVDA,0.0901,0.0951,0.0528,0.0166,0.0295,0.0090,0.0144,0.2597


## Equal-weight portfolio

In [6]:
equal_weights = equal_weight_portfolio(asset_names)
summarize_portfolio("equal_weight", equal_weights, expected_returns, covariance_matrix)

{'portfolio': 'equal_weight',
 'annualized_return': 0.2469403247116475,
 'annualized_volatility': 0.1938337315647533,
 'sharpe_ratio': 1.2739801412178513}

## Minimum-volatility portfolio

In [7]:
minimum_volatility_weights = minimum_volatility_portfolio(expected_returns, covariance_matrix)
summarize_portfolio("minimum_volatility", minimum_volatility_weights, expected_returns, covariance_matrix)

{'portfolio': 'minimum_volatility',
 'annualized_return': 0.13799916006672744,
 'annualized_volatility': 0.15893367578188622,
 'sharpe_ratio': 0.8682814349308298}

## Maximum-Sharpe portfolio

In [8]:
maximum_sharpe_weights = maximum_sharpe_portfolio(expected_returns, covariance_matrix, risk_free_rate=0.0)
summarize_portfolio("maximum_sharpe", maximum_sharpe_weights, expected_returns, covariance_matrix)

{'portfolio': 'maximum_sharpe',
 'annualized_return': 0.4357875989843946,
 'annualized_volatility': 0.278876509091944,
 'sharpe_ratio': 1.5626543820538068}

## Random portfolio simulation

In [9]:
random_portfolios = generate_random_portfolios(
    expected_returns,
    covariance_matrix,
    n_portfolios=5000,
    risk_free_rate=0.0,
    random_state=42,
)
random_portfolios.to_csv(PORTFOLIOS_DIR / "random_portfolios.csv", index=False)
display(random_portfolios.head())
print(f"Random portfolios generated: {len(random_portfolios):,}")

,annualized_return,annualized_volatility,sharpe_ratio,weight_AAPL,weight_MSFT,weight_JPM,weight_PG,weight_XOM,weight_JNJ,weight_KO,weight_NVDA
0,0.332710,0.240819,1.381573,0.178376,0.173330,0.176933,0.020759,0.006413,0.107778,0.104610,0.231802
1,0.164381,0.177816,0.924447,0.013698,0.180788,0.012167,0.188123,0.299077,0.066834,0.212750,0.026563
2,0.267355,0.204531,1.307159,0.014636,0.050373,0.144032,0.066004,0.199360,0.035732,0.293748,0.196115
3,0.365207,0.248326,1.470676,0.159131,0.101207,0.109999,0.018703,0.043588,0.166295,0.094314,0.306763
4,0.215175,0.183360,1.173513,0.182100,0.061154,0.118511,0.164896,0.088374,0.082739,0.225903,0.076323


Random portfolios generated: 5,000


## Efficient frontier-style risk-return analysis

In [10]:
weights_df = pd.DataFrame({
    "asset": asset_names,
    "equal_weight": equal_weights,
    "minimum_volatility": minimum_volatility_weights,
    "maximum_sharpe": maximum_sharpe_weights,
})
weights_df.to_csv(PORTFOLIOS_DIR / "portfolio_weights.csv", index=False)

portfolio_summary = pd.DataFrame([
    summarize_portfolio("equal_weight", equal_weights, expected_returns, covariance_matrix),
    summarize_portfolio("minimum_volatility", minimum_volatility_weights, expected_returns, covariance_matrix),
    summarize_portfolio("maximum_sharpe", maximum_sharpe_weights, expected_returns, covariance_matrix),
])
portfolio_summary.to_csv(PORTFOLIOS_DIR / "portfolio_summary.csv", index=False)

display(weights_df.style.format({"equal_weight": "{:.2%}", "minimum_volatility": "{:.2%}", "maximum_sharpe": "{:.2%}"}))
display(portfolio_summary.style.format({"annualized_return": "{:.2%}", "annualized_volatility": "{:.2%}", "sharpe_ratio": "{:.2f}"}))
LAST_OPTIMIZATION_STATUS

,asset,equal_weight,minimum_volatility,maximum_sharpe
0,AAPL,12.50%,1.21%,0.95%
1,MSFT,12.50%,7.36%,0.18%
2,JPM,12.50%,1.25%,5.31%
3,PG,12.50%,15.66%,0.12%
4,XOM,12.50%,8.61%,6.41%
5,JNJ,12.50%,41.06%,35.78%
6,KO,12.50%,22.70%,2.90%
7,NVDA,12.50%,2.16%,48.35%


,portfolio,annualized_return,annualized_volatility,sharpe_ratio
0,equal_weight,24.69%,19.38%,1.27
1,minimum_volatility,13.80%,15.89%,0.87
2,maximum_sharpe,43.58%,27.89%,1.56


{'minimum_volatility': "scipy_unavailable: ImportError: cannot import name 'Inf' from 'numpy' (/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/__init__.py); used random_search_fallback",
 'maximum_sharpe': "scipy_unavailable: ImportError: cannot import name 'Inf' from 'numpy' (/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numpy/__init__.py); used random_search_fallback"}

## Portfolio weight comparison

In [11]:
max_sharpe_top_asset = weights_df.sort_values("maximum_sharpe", ascending=False).iloc[0]
min_vol_top_asset = weights_df.sort_values("minimum_volatility", ascending=False).iloc[0]
print(f"Largest max-Sharpe allocation: {max_sharpe_top_asset['asset']} ({max_sharpe_top_asset['maximum_sharpe']:.2%})")
print(f"Largest min-vol allocation: {min_vol_top_asset['asset']} ({min_vol_top_asset['minimum_volatility']:.2%})")

Largest max-Sharpe allocation: NVDA (48.35%)
Largest min-vol allocation: JNJ (41.06%)


## Figures

In [12]:
figure_specs = {
    "portfolio_weights_comparison.png": (
        plot_portfolio_weights(weights_df),
        "bar",
        weights_df[["asset", "maximum_sharpe"]].rename(columns={"asset": "label", "maximum_sharpe": "value"}),
    ),
    "efficient_frontier_simulation.png": (
        plot_efficient_frontier_simulation(random_portfolios, portfolio_summary),
        "scatter",
        random_portfolios[["annualized_volatility", "annualized_return"]].copy(),
    ),
    "portfolio_risk_return_comparison.png": (
        plot_portfolio_risk_return(portfolio_summary),
        "scatter",
        portfolio_summary.copy(),
    ),
    "max_sharpe_allocation.png": (
        plot_allocation_pie_or_bar(weights_df, "maximum_sharpe"),
        "bar",
        weights_df[["asset", "maximum_sharpe"]].rename(columns={"asset": "label", "maximum_sharpe": "value"}),
    ),
}

figure_export_methods = {}
for filename, (fig, chart_type, data) in figure_specs.items():
    figure_export_methods[filename] = save_plotly_or_pillow(fig, FIGURES_DIR / filename, chart_type, data)

figure_export_methods

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/plotly/express/_core.py:1992: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  sf: grouped.get_group(s if len(s) > 1 else s[0])


{'portfolio_weights_comparison.png': 'pillow_fallback',
 'efficient_frontier_simulation.png': 'pillow_fallback',
 'portfolio_risk_return_comparison.png': 'pillow_fallback',
 'max_sharpe_allocation.png': 'pillow_fallback'}

## Business interpretation

- Equal weight provides a transparent benchmark allocation across the eight selected assets.
- The minimum-volatility portfolio emphasizes assets that reduced historical portfolio variance under long-only constraints.
- The maximum-Sharpe portfolio emphasizes assets with stronger historical risk-adjusted returns, which can lead to concentration.
- The random portfolio simulation provides an intuitive view of the tradeoff between expected return, volatility, and Sharpe ratio.

## Limitations

- Expected returns and covariance are estimated from historical daily returns and may not persist.
- The risk-free rate is held at zero for simplicity.
- Portfolios are long-only with weights bounded from 0 percent to 100 percent.
- The local SciPy optimizer is unavailable because of a NumPy compatibility issue; this notebook uses a documented deterministic random-search fallback.
- No transaction costs, turnover limits, tax effects, or full out-of-sample backtesting are included in Phase 3.

## Next steps for Phase 4 backtesting

- Convert static allocations into a simple historical portfolio performance comparison.
- Compare portfolio returns, drawdowns, volatility, and benchmark-relative behavior.
- Keep interpretation business-readable and avoid treating historical optimization as a live investment recommendation.